# Is my model supported?

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/noelEOS/tensor-ee/blob/main/notebooks/02_is_my_model_supported.ipynb)

TensorEE runs a model only if it has an exact Earth Engine version (a *building block*) of
every layer in it. This notebook checks a PyTorch model and tells you what is supported, what
is refused, and what to do next.

- **Fully offline:** no Earth Engine account, no requests, no downloads.
- **Random weights are enough:** the check reads the model's layers, not its predictions.
- **To run:** **Runtime → Run all**. Then put your own model in step 6.
- **Examples:** a supported U-Net, two refused models, and attention models such as SegFormer.

In [ ]:
# @title 1. Install
# Installs from the repository, which gets support for more layers between releases. The
# "swin" extra brings segmentation-models-pytorch 0.5.0 and timm 1.0.30, the versions TensorEE
# was checked with.
!pip install --quiet "tensor-ee[swin] @ git+https://github.com/noelEOS/tensor-ee"
print("TensorEE is installed.")

In [ ]:
# @title 2. Set up the checker
# @markdown Loads the checker, and a helper that explains its report in plain words.
from collections import Counter

import torch
import segmentation_models_pytorch as smp

from tensor_ee import check_compatibility
from tensor_ee.converter.convert import convert
from tensor_ee.core.errors import UnsupportedOperation
from tensor_ee.core.grid import Grid
from tensor_ee.smp import adapt_smp_model

torch.set_num_threads(2)
torch.manual_seed(0)              # random weights, the same every run
INPUT_SHAPE = (1, 2, 64, 64)      # the examples' input: (1, bands, height, width)


def explain(report):
    """Say in plain words what a compatibility report found."""
    facts = report.to_dict()
    if facts["status"] == "supported":
        storage = facts["weight_storage"]
        print("SUPPORTED: TensorEE has a building block for every layer in this model.")
        print(f"  {facts['parameter_count']:,} parameters. After BatchNorm is merged into the "
              f"convolutions, {storage['coefficient_count']:,} numbers would be uploaded as weight assets.")
        print(f"  Output for one input: {facts['output_shape']} (batch, classes, height, width).")
        return
    print(f"NOT SUPPORTED (status: {facts['status']}).")
    if not facts["evaluation"]["model_eval"]:
        print("  The model is in training mode: call model.eval() first.")
    if not facts["evaluation"]["float32_parameters_and_buffers"]:
        print("  Some weights are not float32: TensorEE computes in float32 at most.")
    if facts["trace"]["status"] != "success":
        print("  The model's code could not be read:", facts["trace"]["error"])
    reasons = Counter((blocker["code"], blocker["message"]) for blocker in facts["blockers"])
    for (code, message), count in reasons.most_common(8):
        print(f"  {count} x {message}   [{code}]")
    if len(reasons) > 8:
        print(f"  ... and {len(reasons) - 8} more kinds of refusal (report.to_dict()['blockers'] lists them all)")
    if facts["unknowns"]:
        print(f"  {len(facts['unknowns'])} later layers were not checked, because they depend "
              "on the refused ones.")


def check_my_model(model, input_shape):
    """Try the CNN check, then, if that does not support the model, the converter."""
    model = model.eval()
    print("CNN check (U-Nets and other plain CNNs):")
    try:
        if type(model).__module__.startswith("segmentation_models_pytorch"):
            model_for_check = adapt_smp_model(model, input_shape=input_shape)
        else:
            model_for_check = model      # a plain PyTorch CNN is checked as it is
        report = check_compatibility(model_for_check, input_shape)
        explain(report)
        if report.status == "supported":
            return report
    except (ValueError, RuntimeError, UnsupportedOperation) as refusal:
        print("NOT SUPPORTED:", refusal)

    print()
    print("Converter (attention models):")
    height, width = input_shape[2], input_shape[3]
    try:
        try:
            result = convert(model, input_shape, Grid("EPSG:3857", (2.5, 0, 1000, 0, -2.5, 2000),
                                                      width, height), name="my_model")
        except UnsupportedOperation as refusal:
            if "source_grid=None" not in str(refusal):
                raise
            result = convert(model, input_shape, name="my_model")   # a Swin encoder starts on its own grid
        print(result.report.lines()[0])
        print("Every block's verdict: print(MY_RESULT.report.text())")
        return result
    except UnsupportedOperation as refusal:
        print("NOT SUPPORTED:", refusal.code, "-", str(refusal)[:600])
    except Exception as failure:      # for example, torch.export could not read the model
        print("NOT SUPPORTED: the converter could not read this model:", type(failure).__name__, failure)


print("The checker is ready.")

## 3. A supported model: a U-Net with a ResNet-18 encoder

- The same architecture as notebook 01's water model, with random weights.
- `adapt_smp_model` prepares a copy of an SMP model for one input size, and checks that the
  copy gives exactly the original's output on a test input.
- `check_compatibility` then checks every layer.

In [ ]:
unet = smp.Unet("resnet18", encoder_weights=None, in_channels=2, classes=2).eval()
prepared = adapt_smp_model(unet, input_shape=INPUT_SHAPE)
if prepared.tensor_ee_smp_metadata["torch_equal"]:
    print("The prepared copy gives exactly the original's output on a test input.")
explain(check_compatibility(prepared, INPUT_SHAPE))

- **"Supported"** means TensorEE can build the model from its blocks. That is not yet a claim
  about the numbers: whether Earth Engine gives the same as PyTorch is checked on a real run
  (notebook 03, step 10).
- Supported is not accurate: accuracy depends on training.

## 4. Refused models

### A LinkNet: transposed convolutions

- LinkNet's decoder upsamples with `ConvTranspose2d`, which has no building block yet. Support
  for more layers is being added; installing from the repository (step 1) gets the latest.

In [ ]:
linknet = smp.Linknet("resnet18", encoder_weights=None, in_channels=2, classes=2).eval()
explain(check_compatibility(adapt_smp_model(linknet, input_shape=INPUT_SHAPE), INPUT_SHAPE))

### DeepLabV3+: dilated convolutions, and more

- Its encoder uses dilated convolutions. TensorEE's convolutions support stride 1 or 2, no
  dilation, and "same" zero padding.
- The report also names its adaptive average pooling and bilinear resize.

In [ ]:
deeplab = smp.DeepLabV3Plus("resnet18", encoder_weights=None, in_channels=2, classes=2).eval()
explain(check_compatibility(adapt_smp_model(deeplab, input_shape=INPUT_SHAPE), INPUT_SHAPE))

**A refusal is firm:** TensorEE never approximates. It names the layers it cannot run; the
layers after them are not checked. Your options:

- **Change the architecture** (for example a U-Net decoder instead of transposed convolutions)
  and retrain.
- **Ask for the layer:** [open an issue](https://github.com/noelEOS/tensor-ee/issues) with the
  message and your architecture. A new block comes with its own tests and evidence
  ([CONTRIBUTING](https://github.com/noelEOS/tensor-ee/blob/main/CONTRIBUTING.md)).

## 5. Attention models: the converter

- Transformer (attention) models do not go through the CNN check: `adapt_smp_model` refuses
  them and points to a separate tool, the *converter* (`tensor_ee.converter.convert`).
- The converter was built for two models, SegFormer-B0 and a Swin-T UPerNet, on 224 × 224
  inputs.

In [ ]:
segformer = smp.Segformer("mit_b0", encoder_weights=None, in_channels=2, classes=2).eval()
try:
    adapt_smp_model(segformer, input_shape=(1, 2, 224, 224))
except ValueError as refusal:
    print("adapt_smp_model:", refusal)

- To read a model, the converter runs its code on placeholder inputs, so convert only models
  whose source code you trust.
- Its report splits the model into steps (*nodes*). A *validated record* is a set of block
  settings already checked in Earth Engine; each block is **inside** its record or **outside**.
- The first line counts them; the "layout operations" only move pixels into and out of the
  model's own layout.

In [ ]:
chip = Grid("EPSG:3857", (2.5, 0, 1000, 0, -2.5, 2000), 224, 224)   # a placeholder 224 × 224 input area
b0 = convert(segformer, (1, 2, 224, 224), chip, name="segformer_b0")
print(b0.report.lines()[0])

- SegFormer-B0: all 18 blocks inside.
- SegFormer-B1 (the same structure, wider) converts, but most of its blocks are **outside**: TensorEE
  can build them, but they have not been checked in Earth Engine yet. Outside is reported, not
  refused.

In [ ]:
segformer_b1 = smp.Segformer("mit_b1", encoder_weights=None, in_channels=2, classes=2).eval()
b1 = convert(segformer_b1, (1, 2, 224, 224), chip, name="segformer_b1")
print(b1.report.lines()[0])
print(next(node for node in b1.report.nodes if node.id == "encoder.block1.0").verdict())

- A refusal: at 512 × 512, SegFormer-B0's attention works on a grid larger than 7 × 7, which
  needs a building block the library does not have yet.

In [ ]:
try:
    convert(segformer, (1, 2, 512, 512), Grid("EPSG:3857", (2.5, 0, 1000, 0, -2.5, 2000), 512, 512))
except UnsupportedOperation as refusal:
    print("Refused:", refusal.code)
    print(str(refusal)[:400])

## 6. Your model

- Build your model exactly as you trained it (random weights give the same answer), and give
  its input shape. For example:

```python
MY_MODEL = smp.Unet("resnet34", encoder_weights=None, in_channels=4, classes=3)
MY_MODEL.load_state_dict(torch.load("my_checkpoint.pth", map_location="cpu", weights_only=True))
MY_INPUT_SHAPE = (1, 4, 256, 256)
```

- The check tries the CNN check first (SMP models and plain PyTorch CNNs), then the converter.

In [ ]:
MY_MODEL = None                      # change to your torch.nn.Module
MY_INPUT_SHAPE = (1, 2, 224, 224)    # change to (1, bands, height, width)

if MY_MODEL is None:
    print("No model yet: set MY_MODEL above, then run this cell again.")
else:
    MY_RESULT = check_my_model(MY_MODEL, MY_INPUT_SHAPE)

## What next

- **Supported CNN:** [notebook 03](https://github.com/noelEOS/tensor-ee/blob/main/notebooks/03_bring_your_own_cnn.ipynb)
  takes it from checkpoint to an exported region.
- **Converted attention model:** the
  [converter guide](https://github.com/noelEOS/tensor-ee/blob/main/docs/converter.md) explains
  how to run it in Earth Engine.
- **Outside the checked settings:** run it and compare with PyTorch on your input to make that
  evidence; the maintainer can advise.
- **Refused:** see step 4. The
  [block catalogue](https://github.com/noelEOS/tensor-ee/blob/main/docs/building-blocks.md)
  lists every block and its checked settings.